In [1]:
from config import config
from src.data import load_data

train_df, test_df = load_data(config)

print('Train:', train_df.shape)
print('Test:', test_df.shape)

assert train_df.shape == (1460, 81)
assert test_df.shape == (1459, 80)

assert 'SalePrice' in train_df.columns
assert 'SalePrice' not in test_df.columns

assert train_df['Id'].is_unique
assert test_df['Id'].is_unique

assert train_df['SalePrice'].notna().all()
assert (train_df['SalePrice'] > 0).all()

print(train_df['MasVnrType'].value_counts(dropna=False))

Train: (1460, 81)
Test: (1459, 80)
MasVnrType
None       864
BrkFace    445
Stone      128
BrkCmn      15
NaN          8
Name: count, dtype: int64


In [2]:
import pandas as pd
from config import config

# Старый вариант
train_old = pd.read_csv(config.paths.path_to_csv)

# Новый вариант
train_new = pd.read_csv(
    config.paths.path_to_csv,
    keep_default_na=False,
    na_values=['', 'NA'],
)

comparison = pd.concat(
    [
        train_old['MasVnrType'].value_counts(dropna=False),
        train_new['MasVnrType'].value_counts(dropna=False),
    ],
    axis=1,
    keys=['Старое чтение', 'Новое чтение'],
).fillna(0).astype(int)

comparison

,Старое чтение,Новое чтение
MasVnrType,,
NaN,872,8
BrkFace,445,445
Stone,128,128
BrkCmn,15,15
None,0,864


In [3]:
train_df.loc[train_df['MasVnrType'].isna()]

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
234,235,60,RL,NaN,7851,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,216500
529,530,20,RL,NaN,32668,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,3,2007,WD,Alloca,200624
650,651,60,FV,65.0,8125,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2008,WD,Normal,205950
936,937,20,RL,67.0,10083,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,8,2009,WD,Normal,184900
973,974,20,FV,95.0,11639,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,New,Partial,182000
977,978,120,FV,35.0,4274,Pave,Pave,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,11,2007,New,Partial,199900
1243,1244,20,RL,107.0,13891,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2006,New,Partial,465000
1278,1279,60,RL,75.0,9473,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,3,2008,WD,Normal,237000


In [4]:
import numpy as np

X = train_df.drop(columns=['Id', 'SalePrice']).copy()
y = np.log(train_df['SalePrice'])

X_test = test_df.drop(columns=['Id']).copy()

In [5]:
numerical_cols = X.select_dtypes(include='number').columns.tolist()
categorical_cols = X.select_dtypes(exclude='number').columns.tolist()

print('Числовых:', len(numerical_cols))
print('Категориальных:', len(categorical_cols))

print('Числовые признаки:')
print(numerical_cols)

Числовых: 35
Категориальных: 44
Числовые признаки:
['LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold']


## Linear Regression preprocessing checks

In [6]:
from src.train_functions import build_pipeline

pipe = build_pipeline(config)

features_sample = X.iloc[:100]
labels_sample = y.iloc[:100]

# Проверяем обучение и получение предсказаний на небольшой части данных.
pipe.fit(features_sample, labels_sample)

predictions_log = pipe.predict(X.iloc[100:105])
predictions_dollars = np.exp(predictions_log)

assert predictions_log.shape == (5,)
assert np.isfinite(predictions_log).all()
assert (predictions_dollars > 0).all()

# Проверяем размер матрицы после полного preprocessing.
preprocessor = pipe.named_steps['preprocessor']
transformed_sample = preprocessor.transform(features_sample)
feature_names = preprocessor.get_feature_names_out()

assert transformed_sample.shape[0] == len(features_sample)
assert transformed_sample.shape[1] == len(feature_names)
assert np.isfinite(transformed_sample).all()

print('Исходная выборка:', features_sample.shape)
print('После preprocessing:', transformed_sample.shape)

pd.DataFrame({
    'actual_price': train_df['SalePrice'].iloc[100:105].to_numpy(),
    'predicted_price': predictions_dollars,
})

Исходная выборка: (100, 79)
После preprocessing: (100, 200)


c:\Users\aukha\anaconda3\envs\house_prices\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [43] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,actual_price,predicted_price
0,205000,176363.858468
1,178000,220208.486317
2,118964,124892.335360
3,198900,156846.286716
4,169500,161693.340050


In [7]:
structural_transformer = preprocessor.named_steps['structural_missing']
structural_features = structural_transformer.transform(X)

no_pool = X['PoolArea'].eq(0)
no_fireplace = X['Fireplaces'].eq(0)
no_garage = X['GarageCars'].eq(0) & X['GarageArea'].eq(0)
no_basement = X['TotalBsmtSF'].eq(0)

garage_features = [
    'GarageType',
    'GarageFinish',
    'GarageQual',
    'GarageCond',
]

basement_features = [
    'BsmtQual',
    'BsmtCond',
    'BsmtExposure',
    'BsmtFinType1',
    'BsmtFinType2',
]

# Проверяем заполнение categorical structural missing.
assert structural_features.loc[no_pool, 'PoolQC'].eq('NoPool').all()
assert structural_features.loc[no_fireplace, 'FireplaceQu'].eq('NoFireplace').all()
assert structural_features.loc[no_garage, garage_features].eq('NoGarage').all().all()
assert structural_features.loc[no_basement, basement_features].eq('NoBasement').all().all()

assert structural_features['Alley'].notna().all()
assert structural_features['Fence'].notna().all()
assert structural_features['MiscFeature'].notna().all()

# Настоящий пропуск BsmtExposure при существующем подвале должен сохраниться.
unknown_basement_exposure = X['BsmtExposure'].isna() & X['TotalBsmtSF'].gt(0)

assert structural_features.loc[unknown_basement_exposure, 'BsmtExposure'].isna().all()

print('NoPool:', structural_features['PoolQC'].eq('NoPool').sum())
print('NoFireplace:', structural_features['FireplaceQu'].eq('NoFireplace').sum())
print('NoGarage:', structural_features['GarageType'].eq('NoGarage').sum())
print('NoBasement:', structural_features['BsmtQual'].eq('NoBasement').sum())
print('Настоящих пропусков BsmtExposure:', unknown_basement_exposure.sum())

NoPool: 1453
NoFireplace: 690
NoGarage: 81
NoBasement: 37
Настоящих пропусков BsmtExposure: 1


In [10]:
from src.experiment_logging import print_model_diagnostics

for fold, pipe in enumerate(fold_models, start=1):
    preprocessor = pipe.named_steps['preprocessor']
    column_transformer = preprocessor.named_steps['columns']
    model = pipe.named_steps['model']

    selected_columns = [
        column
        for name, transformer, columns in column_transformer.transformers_
        if name in ('numerical', 'categorical')
        for column in columns
    ]

    transformed_feature_count = len(preprocessor.get_feature_names_out())
    matrix_rank = model.rank_
    rank_deficit = transformed_feature_count - matrix_rank

    assert 'Id' not in selected_columns
    assert 'SalePrice' not in selected_columns
    assert len(selected_columns) == 79
    assert transformed_feature_count == model.n_features_in_

    print(
        f'Fold {fold}: исходных features — {len(selected_columns)}, '
        f'после preprocessing — {transformed_feature_count}, '
        f'ранг — {matrix_rank}, '
        f'дефицит ранга — {rank_deficit}'
    )

coefficient_df = print_model_diagnostics(
    fold_models,
    config,
    top_n=20,
)

Fold 1: исходных features — 79, после preprocessing — 266, ранг — 251, дефицит ранга — 15
Fold 2: исходных features — 79, после preprocessing — 263, ранг — 250, дефицит ранга — 13
Fold 3: исходных features — 79, после preprocessing — 269, ранг — 257, дефицит ранга — 12
Fold 4: исходных features — 79, после preprocessing — 263, ранг — 250, дефицит ранга — 13
Fold 5: исходных features — 79, после preprocessing — 262, ранг — 248, дефицит ранга — 14

Linear Regression Diagnostics
Mean intercept: 8.12153
Intercept STD: 1.48759
Transformed features per fold: 262–269
Design matrix rank per fold: 248–257
Top 20 coefficients by mean absolute value:
                               feature  mean_coefficient  mean_abs_coefficient  std_coefficient  fold_count
         categorical__RoofMatl_Membran           2.53480               2.53480          0.81398           4
         categorical__RoofMatl_CompShg           2.24001               2.24001          0.80952           4
           categorical__Roof

In [12]:
def format_dollars(value):
    return f'${value:,.2f}'

oof_analysis = train_cv_df[['Id', 'SalePrice']].copy()

oof_analysis['fold'] = fold_ids + 1
oof_analysis['actual_log'] = np.log(oof_analysis['SalePrice'])
oof_analysis['prediction_log'] = oof_predictions
oof_analysis['prediction_price'] = np.exp(oof_predictions)

oof_analysis['error_log'] = (
    oof_analysis['prediction_log']
    - oof_analysis['actual_log']
)

oof_analysis['abs_error_log'] = oof_analysis['error_log'].abs()
oof_analysis['squared_error_log'] = oof_analysis['error_log'] ** 2

fold_error_summary = (
    oof_analysis
    .groupby('fold')
    .agg(
        samples=('Id', 'size'),
        mae_log=('abs_error_log', 'mean'),
        mean_squared_error_log=('squared_error_log', 'mean'),
        max_abs_error_log=('abs_error_log', 'max'),
    )
)

fold_error_summary['rmse_log'] = np.sqrt(
    fold_error_summary['mean_squared_error_log']
)

fold_error_summary = fold_error_summary[
    [
        'samples',
        'rmse_log',
        'mae_log',
        'max_abs_error_log',
    ]
]

largest_errors = (
    oof_analysis
    .sort_values(
        ['fold', 'abs_error_log'],
        ascending=[True, False],
    )
    .groupby('fold')
    .head(5)
)

display(fold_error_summary.round(5))

largest_errors_display = largest_errors[
    [
        'fold',
        'Id',
        'SalePrice',
        'prediction_price',
        'error_log',
        'abs_error_log',
    ]
]

display(
    largest_errors_display.style.format({
        'SalePrice': format_dollars,
        'prediction_price': format_dollars,
        'error_log': '{:.5f}',
        'abs_error_log': '{:.5f}',
    })
)

influential_houses = oof_analysis[
    oof_analysis['Id'].isin([524, 1299])
]

display(
    influential_houses.style.format({
        'SalePrice': format_dollars,
        'prediction_price': format_dollars,
        'actual_log': '{:.5f}',
        'prediction_log': '{:.5f}',
        'error_log': '{:.5f}',
        'abs_error_log': '{:.5f}',
        'squared_error_log': '{:.5f}',
    })
)

,samples,rmse_log,mae_log,max_abs_error_log
fold,,,,
1,234,0.13292,0.08972,0.98118
2,234,0.25711,0.11673,2.72961
3,234,0.15815,0.08931,1.55727
4,233,0.15531,0.09609,0.96559
5,233,0.22473,0.11098,2.47731


,fold,Id,SalePrice,prediction_price,error_log,abs_error_log
346,1,347,"$151,500.00","$56,792.58",-0.98118,0.98118
529,1,530,"$200,624.00","$316,186.05",0.45490,0.45490
581,1,582,"$253,293.00","$383,718.56",0.41536,0.41536
1270,1,1271,"$260,000.00","$177,685.22",-0.38067,0.38067
313,1,314,"$375,000.00","$534,139.76",0.35373,0.35373
1275,2,1276,"$137,000.00","$8,938.54",-2.72961,2.72961
825,2,826,"$385,000.00","$74,944.27",-1.63650,1.63650
88,2,89,"$85,000.00","$33,234.95",-0.93905,0.93905
632,2,633,"$82,500.00","$174,006.72",0.74630,0.74630
30,2,31,"$40,000.00","$68,454.01",0.53728,0.53728


,Id,SalePrice,fold,actual_log,prediction_log,prediction_price,error_log,abs_error_log,squared_error_log
523,524,"$184,750.00",3,12.12676,13.68403,"$876,798.99",1.55727,1.55727,2.42510
1298,1299,"$160,000.00",5,11.98293,14.46024,"$1,905,475.72",2.47731,2.47731,6.13708


In [9]:
for fold, pipe in enumerate(fold_models, start=1):
    preprocessor = pipe.named_steps['preprocessor']

    selected_columns = [
        column
        for name, transformer, columns in preprocessor.transformers_
        if name in ('numerical', 'categorical')
        for column in columns
    ]

    assert 'Id' not in selected_columns
    assert 'SalePrice' not in selected_columns
    assert len(selected_columns) == 79

    encoded_count = len(preprocessor.get_feature_names_out())
    print(f'Fold {fold}: исходных признаков — {len(selected_columns)}, после one-hot — {encoded_count}')

AttributeError: 'Pipeline' object has no attribute 'transformers_'

In [ ]:
from src.experiment_logging import save_oof_predictions

oof_df = save_oof_predictions(train_cv_df, oof_predictions, fold_ids, config)

assert len(oof_df) == len(train_cv_df)
assert oof_df['Id'].equals(train_cv_df['Id'])
assert np.isfinite(oof_df['prediction_price']).all()

columns = ['Id', 'SalePrice', 'prediction_price', 'error_price', 'abs_error_log']
oof_df.nlargest(10, 'abs_error_log')[columns]

OOF predictions saved: outputs\oof\1_baseline_RF.csv
OOF errors saved: outputs\oof\1_baseline_RF_errors.csv


,Id,SalePrice,prediction_price,error_price,abs_error_log
523,524,184750,528633.237557,343883.237557,1.051291
30,31,40000,96213.470662,56213.470662,0.877690
495,496,34900,82679.688823,47779.688823,0.862487
632,633,82500,192670.137286,110170.137286,0.848181
1298,1299,160000,367912.518981,207912.518981,0.832671
410,411,60000,125606.674511,65606.674511,0.738811
1349,1350,122000,254679.021093,132679.021093,0.735983
462,463,62383,128460.513438,66077.513438,0.722329
916,917,35311,71087.130609,35776.130609,0.699712
1324,1325,147000,285657.232952,138657.232952,0.664360


In [ ]:
from src.train_functions import predict_with_pipeline_ensemble, calculate_regression_metrics
from src.experiment_logging import print_regression_metrics

oof_metrics = calculate_regression_metrics(train_cv_df['SalePrice'], oof_predictions)
print_regression_metrics('OOF', oof_metrics)

features_holdout = holdout_df.drop(columns=['SalePrice'])
holdout_predictions_log = predict_with_pipeline_ensemble(features_holdout, fold_models)

holdout_metrics = calculate_regression_metrics(holdout_df['SalePrice'], holdout_predictions_log)
print_regression_metrics('Holdout', holdout_metrics)


OOF:
RMSE(log): 0.14762
MAE($):    18,324.75
MSE($²):   965,909,905.65
RMSE($):   31,079.09
R²:        0.84792
MAPE:      10.32%
SMAPE:     9.80%
WAPE:      10.04%

Holdout:
RMSE(log): 0.13891
MAE($):    17,035.45
MSE($²):   769,658,563.94
RMSE($):   27,742.72
R²:        0.87328
MAPE:      9.98%
SMAPE:     9.71%
WAPE:      9.78%


In [ ]:
from src.experiment_logging import save_holdout_predictions

holdout_predictions_df = save_holdout_predictions(holdout_df, holdout_predictions_log, config)

display(holdout_predictions_df.head())

Holdout predictions saved: outputs\holdout\1_baseline_RF.csv


,Id,SalePrice,actual_log,prediction_log,prediction_price,error_log,abs_error_log,error_price
1261,1262,128900,11.766792,11.804205,133813.915806,0.037413,0.037413,4913.915806
138,139,230000,12.345835,12.433006,250949.209656,0.087171,0.087171,20949.209656
578,579,146000,11.891362,12.006417,163802.511248,0.115055,0.115055,17802.511248
1125,1126,115000,11.652687,11.694981,119968.058935,0.042293,0.042293,4968.058935
1356,1357,110000,11.608236,11.682233,118448.428652,0.073997,0.073997,8448.428652


In [ ]:
from src.experiment_logging import save_submission

test_predictions_log = predict_with_pipeline_ensemble(test_df, fold_models)
submission_df = save_submission(test_df, test_predictions_log, config)

print('Submission shape:', submission_df.shape)
display(submission_df.head())

Submission saved: outputs\submissions\1_baseline_RF.csv
Submission shape: (1459, 2)


,Id,SalePrice
0,1461,127066.831830
1,1462,154145.209126
2,1463,178352.283953
3,1464,181434.217230
4,1465,191029.506057
